# Lewisham · `status` (web: "Status")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `status` values

A `terms` aggregation on `status.raw` (exact values, case-sensitive) plus a `missing`
aggregation for records with no status. `size: 0` = no records, only the counts.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": lewisham_filter(),
    "aggs": {
        "status": {"terms": {"field": "status.raw", "size": 100}},
        "no_status": {"missing": {"field": "status.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["status"]["buckets"]
status_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_status"]["doc_count"])],
    columns=["status", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Lewisham applications:", total, "| sum of counts:", status_counts["count"].sum())

Lewisham applications: 29021 | sum of counts: 29021


In [3]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "Status" (extracted 2026-11-09).
SCHEMA_STATUS = [
    "Allowed", "Appeal Received", "Application Under Consideration", "Approved",
    "Called in by Secretary of State", "Closed", "Comment Issued", "Declined to Determine",
    "Dismissed", "Appeal In Progress", "Insufficient Fee", "No Objection to Proposal (OBS only)",
    "Not Required", "Objection Raised to Proposal (OBS only)", "Opinion Issued",
    "Pre-application Advice Case Completed", "Refused", "Split Decision", "Unknown", "Withdrawn",
    "Application Invalid", "Application Received", "Quashed", "Commenced", "Completed", "Lapsed",
    "Superseded", "Revoked", "Pending Legacy Record", "Referred to Mayor",
]

status_counts["in_schema"] = status_counts["status"].isin(SCHEMA_STATUS)
status_counts["share_%"] = (100 * status_counts["count"] / total).round(2)
status_counts

,status,count,in_schema,share_%
0,Approved,15062,True,51.90
1,Completed,3699,True,12.75
2,Refused,3689,True,12.71
3,Lapsed,1475,True,5.08
4,NaN,1183,False,4.08
5,Withdrawn,950,True,3.27
6,Not Required,716,True,2.47
7,Application Under Consideration,553,True,1.91
8,Superseded,497,True,1.71
9,Dismissed,446,True,1.54


### Findings · Step 1

- **29,021** Lewisham records (live API, 2026-11-11); the counts add up.
- **1,183 (4.1%) have no status.** Brent had 407 empty records with no status, which
  were dropped; whether these are the same kind of record is not checked yet.
- **Two values not in the schema:**
  - `Appeal in Progress` (1): lower-case "in"; the schema has `Appeal In Progress`. A
    case variant, like the ones merged for Brent.
  - `Historical` (1): not a schema value at all. Not checked yet.
- **No ZZZ.** Brent's 16 deleted appeal records have no counterpart here.
- **No `Application Received`.** Brent has it; Lewisham's pending applications show only as
  `Application Under Consideration` (553).
- **Values Brent never had:** `Not Required` (716) and `Comment Issued` (44), both schema
  values.

## Step 2 · Records with no status: are they empty records?

For Brent, 407 records had no `status`, `decision`, `decision_date` or `description`
(and no borough or URL); they were dropped (`drop_empty_records`). Here we fetch every
Lewisham record with no status (`search_after`, sorted by `id`) and check the same four
fields in pandas, so empty strings count as empty too, plus `borough` and
`url_planning_app`.

In [4]:
FIELDS = ["id", "status", "decision", "decision_date", "description", "borough", "url_planning_app"]


def fetch_all(query: dict) -> pd.DataFrame:
    rows, after = [], None
    while True:
        body = {"size": 1000, "query": query, "_source": FIELDS, "sort": [{"id": "asc"}]}
        if after:
            body["search_after"] = after
        hits = search(body)["hits"]["hits"]
        if not hits:
            return pd.DataFrame(rows, columns=FIELDS)
        rows += [h["_source"] for h in hits]
        after = hits[-1]["sort"]


no_status_query = {"bool": {"filter": [lewisham_filter()], "must_not": [{"exists": {"field": "status.raw"}}]}}
no_status = fetch_all(no_status_query)


def is_empty(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip() == "")


empty = pd.DataFrame({f: is_empty(no_status[f]) for f in FIELDS[2:]})
rule_fields = ["decision", "decision_date", "description"]
no_status["empty_record"] = empty[rule_fields].all(axis=1)

print("Records with no status:", len(no_status))
print("Also no decision, decision_date and description (407-style):", no_status["empty_record"].sum())
print()
print("Empty per field, among records with no status:")
print(empty.sum().to_string())
print()
print("Combinations of the three rule fields (True = empty):")
print(empty[rule_fields].value_counts().to_string())
print()
print("407-style records that also have no borough / no URL:")
print(empty.loc[no_status["empty_record"], ["borough", "url_planning_app"]].sum().to_string())

Records with no status: 1183
Also no decision, decision_date and description (407-style): 1151

Empty per field, among records with no status:
decision            1161
decision_date       1161
description         1155
borough              452
url_planning_app    1151

Combinations of the three rule fields (True = empty):
decision  decision_date  description
True      True           True           1151
False     False          False            13
True      False          False             5
          True           False             5
False     True           False             5
          False          True              4

407-style records that also have no borough / no URL:
borough              452
url_planning_app    1150


### Findings · Step 2

- **1,151 of the 1,183** records with no status are 407-style empty records: they also
  have no decision, decision_date or description. Brent's rule (`drop_empty_records`)
  would match them.
- **Unlike Brent, most have a borough:** only 452 of the 1,151 have no borough. For
  Brent, every one of the 407 had no borough.
- **1,150 of the 1,151 have no URL;** 1 has one. Not checked yet.
- **32 records with no status are not empty:** 13 have decision, decision_date and
  description; the other 19 have some of them. They need their own look before any rule.

## Step 3 · The 32 non-empty records with no status

Fetched again by `id` with more fields, to see whether a status can be derived. Then the
one empty record that has a URL.

In [5]:
MORE = ["id", "lpa_app_no", "application_type", "decision", "decision_date", "valid_date",
        "appeal_decision", "borough", "lpa_name", "bo_system", "url_planning_app", "description"]


def fetch_ids(ids: list[str]) -> pd.DataFrame:
    body = {"size": len(ids), "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": ids}}]}},
            "_source": MORE, "sort": [{"id": "asc"}]}
    return pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=MORE)


non_empty = fetch_ids(no_status.loc[~no_status["empty_record"], "id"].tolist())
non_empty["description"] = non_empty["description"].str.slice(0, 60)
non_empty["url_planning_app"] = non_empty["url_planning_app"].notna()
print(len(non_empty), "records")
print(non_empty.drop(columns=["lpa_name", "borough"]).to_string())
print()
print(non_empty[["borough", "lpa_name", "bo_system"]].value_counts(dropna=False).to_string())

32 records
                        id     lpa_app_no application_type                                 decision decision_date  valid_date appeal_decision        bo_system  url_planning_app                                                   description
0    Lewisham-DC_17_099662   DC/17/099662        All Other                                 Approved    01/05/2020  06/01/2017            None  CSV Data Import             False  Application for the demolition of existing car garage and co
1    Lewisham-DC_21_120320   DC/21/120320   Prior Approval                                      NaN    03/01/2023  17/02/2021            None  CSV Data Import              True  Prior approval for the construction of a single storey rear 
2    Lewisham-DC_21_122223   DC/21/122223        All Other                                      NaN    22/06/2021  15/06/2021            None  CSV Data Import              True  Details submitted in compliance with Condition 3 (constructi
3    Lewisham-DC_22_129646   DC/2

In [6]:
empty_with_url = fetch_ids(no_status.loc[no_status["empty_record"] & ~empty["url_planning_app"], "id"].tolist())
empty_with_url.T

,0
id,Lewisham-DC_24_137429
lpa_app_no,DC/24/137429
application_type,All Other
decision,None
decision_date,None
valid_date,16/09/2024
appeal_decision,None
borough,Lewisham
lpa_name,Lewisham
bo_system,CSV Data Import


### Findings · Step 3

All 32 are `borough` Lewisham, `lpa_name` Lewisham, `bo_system` CSV Data Import. Two groups,
by `lpa_app_no` prefix:

**16 `PRE/` records (2019–2022):** all have a decision and a decision_date. Their
`decision` holds values that are status values elsewhere: Comment Issued (12),
Objection Raised to Proposal (OBS only) (2), Closed (2). The prefix and the descriptions
("Pre-planning application (concept)", "Concept meeting") suggest pre-application advice.
Not confirmed.

**16 `DC/` records:**
- 1 with decision Approved (01/05/2020) and no URL (DC/17/099662).
- 5 Withdrawn, with no decision_date.
- 5 with a decision_date but no decision. DC/26/143311's decision_date (15/04/2026) is
  before its valid_date (20/04/2026).
- 5 with only a valid_date (2024–2025): no decision and no decision_date.

**The one empty record with a URL** (DC/24/137429) has a valid_date (16/09/2024) and a
URL, nothing else, so it looks like the last DC/ group. The empty-record rule does not look at
valid_date; how many of the 1,151 have one is not checked.

## Step 4 · The `Historical` record

`Historical` is not a schema status value (Step 1). Fetch the record with every field it
has, to see what kind of record it is.

In [7]:
body = {"size": 5, "query": {"bool": {"filter": [lewisham_filter(), {"term": {"status.raw": "Historical"}}]}}}
hits = search(body)["hits"]["hits"]
print(len(hits), "record(s)")
record = hits[0]["_source"]
filled = {k: v for k, v in record.items() if v not in (None, "", [], {})}
for k, v in sorted(filled.items()):
    text = str(v)
    print(f"{k}: {text[:200]}")

1 record(s)
application_details: {'existing_uprns': None}
application_type: All Other
bo_system: CSV Data Import
borough: Lewisham
centroid: {'lat': 51.432284, 'lon': -0.019444}
centroid_easting: 537774.0
centroid_northing: 172225.0
decision: Refused
decision_date: 30/10/1996
description: The erection of a single storey club house to includechanging rooms and ancillary facilitiesmeeting room/loungeweather surface pitch enclosed by a 4 to 5 metre high fencefloodlit by lamps on 15 to 16 
id: Lewisham-95_038596
lapsed_date: 30/10/1999
last_synced: 2026-11-11T02:58:33.769Z
last_updated: 2026-11-10T10:00:03.322Z
last_updated_by: CSV Data Import
lpa_app_no: 95/038596
lpa_name: Lewisham
pp_id: LE/247/B/TP
site_name: THE SPORTS GROUNDS RANDLESDOWN ROAD SE6 (BELLINGHAM PLAYING FIELDS
status: Historical
uprn: 200000561114
url_planning_app: https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=_LEWIS_DCAPR_22070&activeTab=summary
valid_date: 28/11/2023


### Findings · Step 4

One record, Lewisham-95_038596 (95/038596), CSV Data Import:

- **An old refused application:** a club house, changing rooms and a floodlit pitch at
  the sports grounds, Randlesdown Road SE6 (Bellingham Playing Fields). Decision Refused,
  decision_date 30/10/1996.
- **Its dates don't fit together:** valid_date 28/11/2023, 27 years after the decision;
  lapsed_date 30/10/1999 (3 years after the decision).
- **Recently touched:** last_updated 2026-11-10.
- It has a portal URL, so a person can check what the portal shows.

## Step 5 · `status` after cleaning

`clean_record` cannot run on Lewisham's rules yet (handover), so the drops and status
rules of `semantic/lewisham/cleaning.yaml` are applied here by hand, reading the ids and
values from the file: drop_test_records, drop_empty_records, drop_portal_removed,
drop_consultation_responses, drop_stale_duplicates; then outcome_from_portal,
appeal_in_progress_case, status_from_decision, empty_to_unknown.

In [8]:
import yaml

rules = yaml.safe_load((PROJECT_ROOT / "semantic/lewisham/cleaning.yaml").read_text())
COLS = ["id", "status", "decision", "decision_date", "description"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]
raw = pd.DataFrame(rows).reindex(columns=COLS)
for c in COLS[1:]:
    raw[c] = raw[c].map(lambda v: v.strip() or None if isinstance(v, str) else None)

dropped = {}
keep = pd.Series(True, index=raw.index)
for rule in rules["all_records"]:
    if rule["action"] != "drop_records":
        continue
    hit = raw["id"].isin(rule.get("ids", []))
    if "when_all_null" in rule:
        hit |= raw[rule["when_all_null"]].isna().all(axis=1)
    hit &= keep
    dropped[rule["id"]] = int(hit.sum())
    keep &= ~hit
clean = raw[keep].copy()
print("Before:", len(raw), "| dropped:", dropped, "| after:", len(clean))

portal = next(r for r in rules["all_records"] if r["id"] == "outcome_from_portal")
fill = clean["id"].isin(portal["outcomes"]) & clean["status"].isna() & clean["decision"].isna()
clean.loc[fill, "status"] = clean.loc[fill, "id"].map(portal["outcomes"])
clean.loc[fill, "decision"] = clean.loc[fill, "id"].map(portal["outcomes"])
applied = {"outcome_from_portal": int(fill.sum())}
for rule in rules["fields"]["status"]:
    if rule["action"] == "map":
        hit = clean["status"] == rule["from"]
        clean.loc[hit, "status"] = rule["to"]
    elif rule["action"] == "copy_when_null":
        hit = clean["status"].isna() & clean[rule["from"]].notna()
        clean.loc[hit, "status"] = clean.loc[hit, rule["from"]]
    elif rule["action"] == "fill_null":
        hit = clean["status"].isna()
        clean.loc[hit, "status"] = rule["to"]
    applied[rule["id"]] = int(hit.sum())
print("Status rules applied:", applied)
after_counts = clean["status"].value_counts().rename_axis("status").reset_index(name="count")
after_counts

Before: 29021 | dropped: {'drop_test_records': 6, 'drop_empty_records': 1151, 'drop_portal_removed': 6, 'drop_consultation_responses': 15, 'drop_stale_duplicates': 78} | after: 27765
Status rules applied: {'outcome_from_portal': 2, 'appeal_in_progress_case': 0, 'status_from_decision': 22, 'empty_to_unknown': 3}


,status,count
0,Approved,15060
1,Completed,3699
2,Refused,3687
3,Lapsed,1475
4,Withdrawn,954
5,Not Required,715
6,Superseded,497
7,Application Under Consideration,481
8,Dismissed,446
9,No Objection to Proposal (OBS only),215


### Findings · Step 5

- **27,765 records after cleaning** (29,021 − 6 test − 1,151 empty − 6 portal-removed − 15
  consultation responses − 78 stale duplicates).
- **Pending: 481** (Application Under Consideration), not 553: the stale duplicates were
  mostly old pending copies of decided applications.
- **Appeal In Progress: 0.** Its one record (Lewisham-22_127022, "Appeal in Progress") is
  a stale duplicate; the current copy says Dismissed. appeal_in_progress_case applies to 0.
- status_from_decision 22, outcome_from_portal 2, empty_to_unknown 3, as expected.
- 20 values after cleaning; the order changes a little (Superseded 497 now above
  pending 481).